# Install

In [4]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install pymupdf PyPDF2


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# %pip install peft transformers accelerate bitsandbytes datasets
%pip install  transformers datasets


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Les 30 premiers documents

In [3]:


import os
import PyPDF2

# 📌 Dossier contenant les fichiers PDF
dossier_pdf = "./dataset"
fichier_sortie = "./sortie.txt"

# 📌 Ouvrir un fichier pour stocker tout le texte extrait
with open(fichier_sortie, "w", encoding="utf-8") as sortie:
    # 📌 Lire tous les fichiers PDF dans le dossier
    for fichier in os.listdir(dossier_pdf):
        if fichier.endswith(".pdf"):  # Vérifier si c'est un PDF
            chemin_pdf = os.path.join(dossier_pdf, fichier)
            
            with open(chemin_pdf, "rb") as pdf_file:
                lecteur_pdf = PyPDF2.PdfReader(pdf_file)
                texte_pdf = ""

                # 📌 Lire toutes les pages du PDF
                for page in range(len(lecteur_pdf.pages)):
                    texte_pdf += lecteur_pdf.pages[page].extract_text() + "\n"

                # 📌 Ajouter le texte extrait au fichier `sortie.txt`
                sortie.write(f"### Contenu du fichier {fichier} ###\n")
                sortie.write(texte_pdf + "\n\n")

print(f"✅ Extraction terminée ! Tout le texte est enregistré dans {fichier_sortie}")


✅ Extraction terminée ! Tout le texte est enregistré dans ./sortie.txt


# Q/R  Chunking

In [4]:
import ollama

with open("sortie.txt", "r", encoding="utf-8") as f:
    lignes = f.readlines()  # Lire toutes les lignes

taille_chunk = 500
chunks = [lignes[i:i + taille_chunk] for i in range(0, len(lignes), taille_chunk)]
print(f"📄 Nombre de chunks : {len(chunks)}")
print(f"📄 Taille du premier chunk : {len(chunks[0])} lignes")


📄 Nombre de chunks : 32
📄 Taille du premier chunk : 500 lignes


In [5]:
text = "".join((chunks[0]))  # Convertir la liste en une seule chaîne de caractères
print(f"📄 Le premier chunk : {len(text)} ")


📄 Le premier chunk : 19773 


In [ ]:
model = "llama2"  # Modèle à utiliser
with open("dataset_llama2.txt", "w", encoding="utf-8") as dataset_file:

    for i, chunk in enumerate(chunks):
        texte_partiel = "".join(chunk)  # Convertir en texte

        prompt = prompt = f"""
        From the following text:
        {texte_partiel}
        
        If possible, try to generate 3 relevant questions and their answers.
        IF you can't generate any questions, please type 'skip'.
        Format the response as:
        
        Q1: [Your question here]
        R1: [Your answer here]
        
        Q2: [Your question here]
        R2: [Your answer here]

        Q3: [Your question here]
        R3: [Your answer here]

        """
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])

        question_reponse = response["message"]["content"]
        
        dataset_file.write(f"### Question/Réponse {i+1} ###\n")
        dataset_file.write(question_reponse + "\n\n")

        print(f"✅ Question/Réponse {i+1} enregistrée dans dataset_llama2.txt")

✅ Question/Réponse 1 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 2 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 3 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 4 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 5 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 6 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 7 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 8 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 9 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 10 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 11 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 12 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 13 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 14 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 15 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 16 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 17 enregistrée dans dataset_llama2.txt
✅ Question/Réponse 18 e

In [ ]:
import ollama

models = ["llama2", "mistral", "gemma"]

dataset_file = "dataset_llama2.txt"

with open(dataset_file, "r", encoding="utf-8") as file:
    dataset_content = file.readlines()

results = {}

for model in models:
    print(f" Test du modèle : {model}")
    results[model] = []

    for i, line in enumerate(dataset_content):
        if line.startswith("Q"):  # Détecter les questions
            question = line.strip()

            response = ollama.chat(model=model, messages=[{"role": "user", "content": question}])

            answer = response["message"]["content"]
            results[model].append((question, answer))

output_file = "llm_comparaison.txt"

with open(output_file, "w", encoding="utf-8") as out:
    for model, qa_pairs in results.items():
        out.write(f"\n### Résultats du modèle {model} ###\n")
        for q, a in qa_pairs:
            out.write(f"{q}\n{a}\n\n")

print(f" Comparaison terminée ! Résultats enregistrés dans {output_file}")


🔍 Test du modèle : llama2
🔍 Test du modèle : mistral
🔍 Test du modèle : gemma
✅ Comparaison terminée ! Résultats enregistrés dans llm_comparaison.txt


# BertScore

In [8]:
%pip install bert-score
# %pip install  numpy scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import re

# Fonction pour lire le fichier dataset.txt
def load_dataset(file_path):
    # Lire le contenu du fichier
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Expression régulière pour capturer toutes les sections de type ### Question/Réponse X ###
    pattern_section = r'### Question/Réponse \d+ ###\n(.*?)(?=\n### Question/Réponse \d+ ###|\Z)'
    sections = re.findall(pattern_section, text, re.DOTALL)

    answers = {}
    
    # Expression régulière pour détecter chaque question et réponse dans une section
    pattern = r'(Q\d+: (.*?)\nR\d+: (.*?))\n'
    
    for section in sections:
        matches = re.findall(pattern, section)
        # print(matches)
        for match in matches:
            # print(match)
            question = match[0].split('\n')[0]  # Capture la question
            # print(question)
            # print('*************************')
            answer = match[2].strip()     # Capture la réponse
            answers[question] = answer
    
    return answers

# Exemple d'utilisation
file_path = 'dataset_llama2.txt'
q_a = load_dataset(file_path)
print(q_a.keys())
print(q_a.values())
# Affichage du dictionnaire
# for question, answer in q_a.items():
#     print(f'{question} ')


dict_keys(['Q1: What are the expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q2: What is the reporting preview pane showing for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q3: Are there any discrepancies in the acquisition of data during the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q1: What are the limitations of the UFED InField Kiosk v7.5.0.875 in terms of acquiring PIM data from iOS devices?', 'Q2: What are the reporting features available in UFED InField Kiosk v7.5.0.875?', 'Q3: What is the equipment and user data that UFED InField Kiosk v7.5.0.875 can extract?', 'Q1: What is the purpose of the Digital Evidence (DE) standard?', 'Q2: What are the key characteristics of a Digital Evidence Specialist (DES)?', 'Q3: What is the importance of repeatability in digital evidence handling?', 'Q1: How does the ISO 27037 sta

In [10]:
import re

def extract_answers_from_file(file_path, model_name):
    # Lire le contenu du fichier
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Expression régulière pour capturer la section du modèle spécifié
    pattern_section = rf'### Résultats du modèle {model_name} ###\n(.*?)(?=\n### Résultats du modèle \w+ ###|\Z)'
    section = re.search(pattern_section, text, re.DOTALL)

    if section:
        section_text = section.group(1)
        # Expression régulière pour détecter les questions et leurs réponses dans la section
        pattern = r'(Q[1-7]: .*?)\n(.*?)(?=\nQ[1-7]:|$)'
        matches = re.findall(pattern, section_text, re.DOTALL)
        
        answers = {}
        for match in matches:
            question = match[0].strip()
            answer = match[1].strip()
            answers [question] = answer
        return answers
    else:
        return []

# Exemple d'utilisation
file_path = 'llm_comparaison.txt'

# Extraction des réponses pour llama2
llama2_answers = extract_answers_from_file(file_path, model_name="llama2")
# Affichage du dictionnaire
for question, answer in llama2_answers.items():
    print(f'{question}\n')

Q1: What are the expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?

Q2: What is the reporting preview pane showing for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?

Q3: Are there any discrepancies in the acquisition of data during the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?

Q1: What are the limitations of the UFED InField Kiosk v7.5.0.875 in terms of acquiring PIM data from iOS devices?

Q2: What are the reporting features available in UFED InField Kiosk v7.5.0.875?

Q3: What is the equipment and user data that UFED InField Kiosk v7.5.0.875 can extract?

Q1: What is the purpose of the Digital Evidence (DE) standard?

Q2: What are the key characteristics of a Digital Evidence Specialist (DES)?

Q3: What is the importance of repeatability in digital evidence handling?

Q1: How does the ISO 27037 standard address the issue of cha

In [11]:
import re
from bert_score import score
import numpy as np

# Charger les données de référence et des modèles
# reference_dict = load_dataset('dataset_old.txt')
reference_dict = load_dataset('dataset_llama2.txt')

model1_dict = extract_answers_from_file('llm_comparaison.txt', 'llama2')
model2_dict = extract_answers_from_file('llm_comparaison.txt', 'mistral')
model3_dict = extract_answers_from_file('llm_comparaison.txt', 'gemma')

# print(reference_dict.keys())

# Supprimer directement les clés qui ne sont pas dans model1_dict
# reference_dict = {key: value for key, value in reference_dict.items() if key in model1_dict}
model1_dict  = {key: value for key, value in reference_dict.items() if key in reference_dict}
model2_dict  = {key: value for key, value in reference_dict.items() if key in reference_dict}
model3_dict  = {key: value for key, value in reference_dict.items() if key in reference_dict}

print(len(reference_dict.keys()))
print(len(model1_dict.keys()))
print(len(model2_dict.keys()))
print(len(model3_dict.keys()))

print(reference_dict.keys())
print(model1_dict.keys())
print(model2_dict.keys())
print(model3_dict.keys())

# Textes de référence multiples
references = list(reference_dict.values())

# Générations des LLMs pour chaque référence
llm1_responses = list(model1_dict.values())
llm2_responses = list(model2_dict.values())
llm3_responses = list(model3_dict.values())


# Calculer les scores pour chaque LLM
llm1_f1_scores = []
llm2_f1_scores = []
llm3_f1_scores = []
llm1_r1_scores = []
llm2_r1_scores = []
llm3_r1_scores = []
llm1_p1_scores = []
llm2_p1_scores = []
llm3_p1_scores = []

for i, generated in enumerate(llm1_responses):
    P, R, F1 = score([generated], [references[i]], lang='en', verbose=False, batch_size=1)
    llm1_f1_scores.append(F1.item())
    llm1_r1_scores.append(R.item())
    llm1_p1_scores.append(P.item())

for i, generated in enumerate(llm2_responses):
    P, R, F1 = score([generated], [references[i]], lang='en', verbose=False, batch_size=1)
    llm2_f1_scores.append(F1.item())
    llm2_r1_scores.append(R.item())
    llm2_p1_scores.append(P.item())

for i, generated in enumerate(llm3_responses):
    P, R, F1 = score([generated], [references[i]], lang='en', verbose=False, batch_size=1)
    llm3_f1_scores.append(F1.item())
    llm3_r1_scores.append(R.item())
    llm3_p1_scores.append(P.item())

# Calculer les moyennes des scores F1
mean_llm1_f1 = np.mean(llm1_f1_scores)
mean_llm2_f1 = np.mean(llm2_f1_scores)
mean_llm3_f1 = np.mean(llm3_f1_scores)
mean_llm1_r1 = np.mean(llm1_r1_scores)
mean_llm2_r1 = np.mean(llm2_r1_scores)
mean_llm3_r1 = np.mean(llm3_r1_scores)
mean_llm1_p1 = np.mean(llm1_p1_scores)
mean_llm2_p1 = np.mean(llm2_p1_scores)
mean_llm3_p1 = np.mean(llm3_p1_scores)

# Afficher les moyennes des scores F1
print(f"Moyenne F1 pour llama: {mean_llm1_f1:.4f}")
print(f"Moyenne F1 pour mistral: {mean_llm2_f1:.4f}")
print(f"Moyenne F1 pour gemma: {mean_llm3_f1:.4f}")
print(f"Moyenne Recall pour llama: {mean_llm1_r1:.4f}")
print(f"Moyenne Recall pour mistral: {mean_llm2_r1:.4f}")
print(f"Moyenne Recall pour gemma: {mean_llm3_r1:.4f}")
print(f"Moyenne Precision pour llama: {mean_llm1_p1:.4f}")
print(f"Moyenne Precision pour mistral: {mean_llm2_p1:.4f}")
print(f"Moyenne Precision pour gemma: {mean_llm3_p1:.4f}")


# Créer une chaîne de caractères avec les résultats
results_text = (
    f"Moyenne pour llama2 -->  F1: {mean_llm1_f1:.4f}\t R: {mean_llm1_r1}\t P: {mean_llm1_p1}\n"
    f"Moyenne pour mistral --> F1: {mean_llm2_f1:.4f}\t R: {mean_llm2_r1}\t P: {mean_llm2_p1}\n"
    f"Moyenne pour gemma -->   F1: {mean_llm3_f1:.4f}\t R: {mean_llm3_r1}\t P: {mean_llm3_p1}\n\n"
)

# Enregistrer les résultats dans un fichier texte
with open("resultats_llm.txt", "w") as f:
    f.write(results_text)

print("Les résultats ont été enregistrés dans le fichier resultats_llm.txt")


d:\DIC3\nlpCours\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


95
95
95
95
dict_keys(['Q1: What are the expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q2: What is the reporting preview pane showing for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q3: Are there any discrepancies in the acquisition of data during the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q1: What are the limitations of the UFED InField Kiosk v7.5.0.875 in terms of acquiring PIM data from iOS devices?', 'Q2: What are the reporting features available in UFED InField Kiosk v7.5.0.875?', 'Q3: What is the equipment and user data that UFED InField Kiosk v7.5.0.875 can extract?', 'Q1: What is the purpose of the Digital Evidence (DE) standard?', 'Q2: What are the key characteristics of a Digital Evidence Specialist (DES)?', 'Q3: What is the importance of repeatability in digital evidence handling?', 'Q1: How does the I

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

Moyenne F1 pour llama: 1.0000
Moyenne F1 pour mistral: 1.0000
Moyenne F1 pour gemma: 1.0000
Moyenne Recall pour llama: 1.0000
Moyenne Recall pour mistral: 1.0000
Moyenne Recall pour gemma: 1.0000
Moyenne Precision pour llama: 1.0000
Moyenne Precision pour mistral: 1.0000
Moyenne Precision pour gemma: 1.0000
Les résultats ont été enregistrés dans le fichier resultats_llm.txt


# RAG Simple

In [2]:
%pip install sentence-transformers tiktoken 


  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install chromadb 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import chromadb
from sentence_transformers import SentenceTransformer
import tiktoken

client = chromadb.Client()

# Initialise une base persistante
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection(name="rag_collection")
    print(" Collection 'rag_collection' supprimée.")
except Exception as e:
    print(" Aucune collection à supprimer :", e)

collection = client.create_collection(name="rag_collection")
# collection.delete(ids=None)  # Supprime toutes les données


# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def chunk_text(text, max_tokens=4700):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    print(f" Nombre de tokens : {len(tokens)}")
    print(tokens)
    
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i:i + max_tokens]
        chunk_text = tokenizer.decode(chunk)
        chunks.append(chunk_text)
    
    return chunks

file_path = "sortie.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_text(text)

print(f" {len(chunks)} chunks créés")

for i, chunk in enumerate(chunks):
    # embedding = model.encode(chunk).tolist()
    # print(f"Chunk {i+1} - Embedding : {(embedding)}")
    collection.add(
        ids=[str(i)], 
        documents=[chunk], 
        # metadatas=[{"text": chunk}]
    )

print(f" {len(chunks)} chunks indexés dans ChromaDB !")



 Aucune collection à supprimer : Collection rag_collection does not exist.
 Nombre de tokens : 144019
[14711, 2140, 1509, 3930, 45420, 1296, 8234, 65859, 15280, 6239, 582, 17647, 14506, 12, 1739, 31501, 2630, 74, 76017, 2325, 22, 13, 20, 13, 15, 13, 17419, 16378, 66387, 2323, 5832, 369, 13716, 14227, 6515, 62604, 275, 822, 308, 13782, 11, 720, 21180, 1507, 763, 1915, 735, 76017, 348, 22, 13, 20, 13, 15, 13, 17419, 720, 2323, 18591, 369, 13716, 14227, 6515, 9383, 275, 822, 308, 13782, 198, 30649, 220, 1544, 1174, 220, 679, 220, 24, 720, 2028, 436, 4248, 269, 259, 289, 439, 550, 4248, 1636, 369, 259, 568, 6011, 315, 473, 8019, 18615, 8398, 320, 35, 12228, 8, 328, 272, 1873, 323, 12053, 94466, 320, 50, 31389, 8, 555, 720, 1820, 8410, 315, 7658, 44056, 35653, 315, 279, 5165, 10181, 315, 35653, 323, 12053, 13, 2355, 2520, 198, 5217, 2038, 922, 14529, 73703, 328, 31389, 62542, 7224, 11, 4587, 348, 374, 275, 279, 294, 5104, 198, 8198, 323, 5557, 198, 11377, 655, 198, 4868, 13096, 3997, 627, 7

In [8]:
print(len(chunks[0]))

18787


# Recherche dans ChromaDB. 

In [17]:
reference_dict = load_dataset('dataset_llama2.txt')
print(list(reference_dict.keys()))

['Q1: What are the expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q2: What is the reporting preview pane showing for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q3: Are there any discrepancies in the acquisition of data during the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875?', 'Q1: What are the limitations of the UFED InField Kiosk v7.5.0.875 in terms of acquiring PIM data from iOS devices?', 'Q2: What are the reporting features available in UFED InField Kiosk v7.5.0.875?', 'Q3: What is the equipment and user data that UFED InField Kiosk v7.5.0.875 can extract?', 'Q1: What is the purpose of the Digital Evidence (DE) standard?', 'Q2: What are the key characteristics of a Digital Evidence Specialist (DES)?', 'Q3: What is the importance of repeatability in digital evidence handling?', 'Q1: How does the ISO 27037 standard addr

In [ ]:
contexts_rag_simple = []

query = list(reference_dict.keys())[0]
print(f"Query: {query} \n")
results = collection.query(
    query_texts=[query], # Chroma will embed this for you
    n_results=10 # how many results to return        
)
context = results["documents"][0][0]
contexts_rag_simple.append(context)
print(contexts_rag_simple[0])  

Query: Q1: What are the expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875? 

### Contenu du fichier testresultsnistmobiledeviceacquisitiontool-ufedinfieldkiosk_v7.5.0.875.pdf ###
Test Result for Mobile Device Acqusitio n Tool, 
UFED InField Kiosk v7.5.0.875 
Test Results for Mobile Device Acquisitio n Tool
September 27 , 201 9 
This r epor t w as pr epared for t he Department of H omeland Security (DHS) S cience and Technology Directorate (S&T) by 
the Office of Law Enforcement Standards of the National Institute of Standards and Technology.  
For
 additional information about ongoing DHS S&T cybersecurity projects, please v isit the dhs
 science and technology
cyber
 security division website.
 
 September  2019 
Test Results for Mobile Device Acquisition Tool :  
UFED InField  Kiosk v7.5 .0.875   
iiContents  
Introduction  ................................................................ ..................................

In [ ]:
# result = collection.get(ids=["5"])
# result

In [20]:
contexts_rag_simple.__len__()

1

In [ ]:
from IPython.display import display, HTML
import ollama
import textwrap

responses_rag_simple = []
model = "gemma"
# for context, query in zip(contexts_rag_simple, list(reference_dict.keys())):
prompt = f"""
Based on the following context:

{context}

Answer the following question:
{query}
"""

response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])

display(HTML(f'''
<div style="display: flex; align-items: center; gap: 10px;">
  <svg width="20" height="20" viewBox="0 0 24 24" fill="white" xmlns="http://www.w3.org/2000/svg">
    <path d="M12 12c2.7 0 5-2.3 5-5s-2.3-5-5-5-5 2.3-5 5 2.3 5 5 5zm0 2c-3.3 0-10 1.7-10 5v3h20v-3c0-3.3-6.7-5-10-5z"/>
  </svg>
  <p style="color:white; margin:0; font-size:16px;">{query}</p>
</div>
'''))

content = response["message"]["content"]

# Découper le texte après 120 caractères
#  7. Génération avancée avec Gemma
print(f"\n 🤖 Réponse générée du modèle {model} :\n")
response_rag_simple = textwrap.fill(content, width=120)
responses_rag_simple.append(response_rag_simple)

print(responses_rag_simple[0])




 🤖 Réponse générée du modèle gemma :

**Expected results for the internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875:**  -
Successful acquisition of all supported data categories, with the following exceptions:     - Graphic files associated
with contacts are reported separately under the multimedia category.     - Memos are not reported for the Galaxy S5, LG
G4, Galaxy S6 Edge Plus, and Galaxy Note 3.     - Long memos are truncated for the LG G5.     - Call logs status for
incoming and missed calls is incorrectly reported as outgoing for the LG G5.     - Documents (txt, pdf) are not reported
for the Galaxy S5, LG G4, LG G5, GS7 Edge, HTC 10, and Galaxy Note 3.     - SMS and MMS messages are partially reported
for the Ellipsis 8.     - Audio attachments for outgoing MMS files are partially reported with the associated text for
the LG G4.     - Instagram data was not reported for the Samsung J3.     - GPS related data is not reported for
associated M

# RAG Avancé 

In [22]:
%pip install rank_bm25


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import ollama

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:

# query = "How does the tool handle acquisition of PIM data (contacts, calendar, memos/notes) for iOS devices?"

contexts_rag_avancee = []

# for i in list(reference_dict.keys()):
#     query = f"{i}"
results = collection.query(
    query_texts=[query], # Chroma will embed this for you
    n_results=5 # how many results to return
    
)
context = results["documents"][0][0]
contexts_rag_avancee.append(context)
print(contexts_rag_avancee[0])  


### Contenu du fichier testresultsnistmobiledeviceacquisitiontool-ufedinfieldkiosk_v7.5.0.875.pdf ###
Test Result for Mobile Device Acqusitio n Tool, 
UFED InField Kiosk v7.5.0.875 
Test Results for Mobile Device Acquisitio n Tool
September 27 , 201 9 
This r epor t w as pr epared for t he Department of H omeland Security (DHS) S cience and Technology Directorate (S&T) by 
the Office of Law Enforcement Standards of the National Institute of Standards and Technology.  
For
 additional information about ongoing DHS S&T cybersecurity projects, please v isit the dhs
 science and technology
cyber
 security division website.
 
 September  2019 
Test Results for Mobile Device Acquisition Tool :  
UFED InField  Kiosk v7.5 .0.875   
iiContents  
Introduction  ................................................................ .....................................................................  1
How to Read This Report  ................................ ...........................................

In [ ]:
import textwrap

responses_rag_avancee = []
model = "llama2"

# for query, context in zip(list(reference_dict.keys()), contexts_rag_avancee):

# 🔎 5. Re-ranking avec BM25 (pour améliorer la pertinence des résultats)
bm25 = BM25Okapi(context)
scores = bm25.get_scores(query.split())
ranked_results = sorted(zip(context, scores), key=lambda x: x[1], reverse=True)
print(ranked_results.__len__())

top_context = "\n".join([result[0][0] for result in ranked_results])

prompt = f"""
Based on the following context:

{top_context}

Answer the following question:
{query}
"""

response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
content = response['message']['content']


# Découper le texte après 120 caractères
response_rag_avancee = textwrap.fill(content, width=120)
responses_rag_avancee.append(response_rag_avancee)

print("\n🤖 Réponse générée  :\n")
print(responses_rag_avancee[0])




18787

🤖 Réponse générée  :

The information you provided is not clear or specific enough for me to provide a definitive answer to your question. The
expected results for an internal memory acquisition on the Android platform using UFED InField Kiosk v7.5.0.875 can vary
depending on several factors such as:  1. The type of device being used: Different devices have different types of
internal storage, and the expected results may differ depending on the device's storage capacity. For example, a high-
end smartphone may have more internal storage than a basic feature phone. 2. The version of Android being used: The
expected results may differ based on the version of Android being used. Different versions of Android may have different
behaviors or limitations regarding internal storage. 3. The type of app being developed: The expected results may differ
based on the type of app being developed. For example, an app that requires a lot of internal storage may have different
expected results

# Etude comparative (LLM simple, LLM+RAG simple, LLM+RAG avance) Mtrique: Bertscore 

In [26]:

reference_dict = load_dataset('dataset_llama2.txt')

model1_dict = extract_answers_from_file('llm_comparaison2.txt', 'llama2')

# print(reference_dict.keys())

# Supprimer directement les clés qui ne sont pas dans model1_dict
reference_dict = {key: value for key, value in reference_dict.items() if key in model1_dict}



references = list(reference_dict.values())

# Générations des LLMs pour chaque référence
llm1_responses = list(model1_dict.values())

In [ ]:
from bert_score import score
import numpy as np
import csv



llm1_f1_scores = []
rag_simple_f1 = []
rag_avancee_f1 = []

for llm_response, reference in zip(llm1_responses, references):
    P, R, F1 = score([llm_response], [reference], lang='en', verbose=False, batch_size=1)
    llm1_f1_scores.append(F1.item())

for llm_response, reference in zip(responses_rag_simple, references):
    P, R, F1 = score([llm_response], [reference], lang='en', verbose=False, batch_size=1)
    rag_simple_f1.append(F1.item())

for llm_response, reference in zip(responses_rag_avancee, references):
    P, R, F1 = score([llm_response], [reference], lang='en', verbose=False, batch_size=1)
    rag_avancee_f1.append(F1.item())

# Calculer les moyennes des scores F1
mean_llm1_f1 = np.mean(llm1_f1_scores)
mean_rag_simple_f1 = np.mean(rag_simple_f1)
mean_rag_avancee_f1 = np.mean(rag_avancee_f1)




# # Évaluer avec BERTScore
print("\n🔹 BERTScore (LLM Simple) :", mean_llm1_f1)

print("🔹 BERTScore (RAG Simple) :", mean_rag_simple_f1)

print("🔹 BERTScore (RAG Avancé) :", mean_rag_avancee_f1)


file_name = 'bert_scores_llm_ragSimple_ragAvancee.csv'

with open(file_name, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    # Écrire l'en-tête
    writer.writerow(["Type de Modèle", "Scores F1 individuels", "Moyenne F1"])
    
    # Écrire les scores individuels et les moyennes
    writer.writerow(["LLM Simple", llm1_f1_scores, mean_llm1_f1])
    writer.writerow(["RAG Simple", rag_simple_f1, mean_rag_simple_f1])
    writer.writerow(["RAG Avancé", rag_avancee_f1, mean_rag_avancee_f1])

print(f"\n✅ Résultats enregistrés dans '{file_name}'")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro


🔹 BERTScore (LLM Simple) : 0.7930440545082093
🔹 BERTScore (RAG Simple) : 0.7853740453720093
🔹 BERTScore (RAG Avancé) : 0.805726170539856

✅ Résultats enregistrés dans 'bert_scores_llm_ragSimple_ragAvancee.csv'


# RAG+fine tuning

In [1]:
import re
import json

def convert_txt_to_valid_json(txt_file_path, json_file_path):
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Capture les paires Q/R
    blocks = re.findall(r'(Q\d+: .*?)\n(R\d+: .*?)(?=\nQ\d+:|\n###|\Z)', text, re.DOTALL)

    data = []
    for q, r in blocks:
        instruction = q.strip()
        output = r.strip()
        data.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })

    # Sauvegarder au format JSON bien formé
    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f" {len(data)} paires Q/R enregistrées dans {json_file_path} au format JSON valide.")


# Utilisation :
convert_txt_to_valid_json("dataset_llama2.txt", "dataset_llama2.json")

 96 paires Q/R enregistrées dans dataset_llama2.json au format JSON valide.


In [2]:
%pip install transformers datasets accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import json

# 1. Charger les données
with open("qa_dataset.json", "r") as f:
    raw_data = json.load(f)

# 2. Convertir en dataset Hugging Face
data = [{"input": item["instruction"], "output": item["output"]} for item in raw_data]
dataset = Dataset.from_list(data)



d:\DIC3\nlpCours\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset

Dataset({
    features: ['input', 'output'],
    num_rows: 96
})

In [ ]:
from huggingface_hub import login
login("")  # colle ton token ici entre les guillemets


In [5]:
%pip install sentencepiece


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
#  Tokenizer et modèle
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [57]:


# 4. Tokenisation
def preprocess(example):
    input = tokenizer(example["input"], padding="max_length", truncation=True, max_length=256)
    output = tokenizer(example["output"], padding="max_length", truncation=True, max_length=256)
    input["labels"] = output["input_ids"]
    return input

tokenized_dataset = dataset.map(preprocess)




Map: 100%|██████████| 96/96 [00:00<00:00, 676.63 examples/s]


In [58]:
# 5. Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./flan-t5-qa",
    per_device_train_batch_size=8,
    num_train_epochs=10,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch"
    # evaluation_strategy="no"
)

In [59]:
# 6. Entraîneur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

C:\Users\sirou\AppData\Local\Temp\ipykernel_230732\4076080842.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [60]:
trainer.train()

Step,Training Loss
10,33.180100
20,26.848900
30,21.735900
40,17.394000
50,14.116400
60,10.708500
70,8.817100
80,7.375200
90,6.970200
100,6.511800


TrainOutput(global_step=120, training_loss=13.83447748819987, metrics={'train_runtime': 839.3754, 'train_samples_per_second': 1.144, 'train_steps_per_second': 0.143, 'total_flos': 89227442257920.0, 'train_loss': 13.83447748819987, 'epoch': 10.0})

In [ ]:
# import shutil
# import os

# if os.path.exists("./flan-t5-qa"):
#     shutil.rmtree("./flan-t5-qa")


In [61]:
# 7. Sauvegarde
model.save_pretrained("./flan-t5-qa") 
tokenizer.save_pretrained("./flan-t5-qa")

('./flan-t5-qa\\tokenizer_config.json',
 './flan-t5-qa\\special_tokens_map.json',
 './flan-t5-qa\\spiece.model',
 './flan-t5-qa\\added_tokens.json')

In [62]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("./flan-t5-qa")
tokenizer = T5Tokenizer.from_pretrained("./flan-t5-qa")
 


In [63]:
question = "What is the purpose of logging information in X-Ways Forensics?"

inputs = tokenizer(question, return_tensors="pt")
output_ids = model.generate(**inputs, max_length=256)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


Conservation of Nature


In [64]:
contexts_rag_finetuning = []
 

query = "What is the purpose of logging information in X-Ways Forensics?"
print(f"Query: {query} \n")
results = collection.query(
    query_texts=[query], # Chroma will embed this for you
    n_results=10 # how many results to return        
)
context = results["documents"][0][0]
contexts_rag_finetuning.append(context)
print(contexts_rag_finetuning[0])  

Query: What is the purpose of logging information in X-Ways Forensics? 

-83. 
Chung, H., Park, J., Lee, S., Kang, C. (2012) “Digital forensic investigation of cloud storage services”, Digital Investigation , 
Vol. 9, no. 2, pp 81-95. DOI: 10.1016/j.diin.2012.05.015 
Federici, C. (2014) “Cloud Data Imager: A unified answer to remote acquisition of cloud storage areas”, Digital 
Investigation , Vol. 11, no. 1, pp 30-42. DOI: 10.1016/j.diin.2014.02.002 
Hegarty, R., Lamb, D., Attwood, A. (2014) “Digital Evidence Challenges in the Internet of Things”, 9th International 
Workshop on Digital Forensics and Incident Analysis , Plymouth, pp 163–172. 
ISO. (2012) ISO/IEC 27037:2012 Information Technology — Security Techniques — Guidelines for Identification, Collection, 
Acquisition, and Preservation of Digital Evidence , International Organization for Standardization, Geneva. 
Sindhu, K.K., Meshram, B.B. (2012) “Digital Forensic Investigation Tools and Procedures”, International Journal of Com

In [65]:
def generate_answer(question, context):
    # Étape 1 : récupérer le contexte
    context = context
    
    # Étape 2 : construire le prompt
    prompt = f"Contexte : {context}\n\nQuestion : {question}\n\nRéponse :"

    # Étape 3 : tokenizer et générer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_length=256)
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return answer

In [66]:
print(generate_answer(query, contexts_rag_finetuning[0]))


Digital forensic investigation of cloud storage services”, Digital Investigation , Vol. 9, no. 2, pp 81-95. DOI: 10.1016/j.diin.2012.05.015 Federici, C. (2014) “Cloud Data Imager: A unified answer to remote acquisition of cloud storage areas”, Digital Investigation , Vol. 11, no. 1, pp 30-42. DOI: 10.5815/ijcnis.2012.04.05
